# Health-LLM PMData Stress Workflow

This notebook demonstrates the end-to-end workflow using PMData for the stress estimation task:

1) Load PMData (CSV/JSON) for a participant
2) Build prompts (instruction, input, output)
3) Tokenize using the repo's `DataHandler` with the medAlpaca template
4) Prepare a small `datasets.Dataset` and wire up a minimal HF `Trainer`

We show intermediate outputs: sample prompts, labels, tokenized shapes, model and trainer info.

Note: Running training with large models (e.g., LLaMA) requires access to weights and significant GPU resources. Here we default to a tiny model for demonstration, but you can switch to your target model easily.

In [1]:
# Imports
import os, json, csv, math
from datetime import datetime, timedelta
from typing import List, Dict, Any

import numpy as np
from datasets import Dataset

import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForSeq2Seq, Trainer, TrainingArguments
)

import sys
from pathlib import Path
# Ensure repo root is on sys.path and as CWD so relative data paths work
_here = Path().resolve()
for _cand in [_here, _here.parent, _here.parent.parent]:
    if (_cand / 'medalpaca' / 'handler.py').exists():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        try:
            os.chdir(_cand)
        except Exception:
            pass
        break
from medalpaca.handler import DataHandler

print('Torch CUDA available:', torch.cuda.is_available())
print('Working directory:', os.getcwd())

/home/hshi/anaconda3/envs/healthllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch CUDA available: True
Working directory: /home/hshi/Documents/MyRepos/Health-LLM-copied


In [ ]:
import torch

: 

### Hugging Face access for Gemma-3-270M
- Accept the model license at `https://huggingface.co/google/gemma-3-270m`.
- Login locally with `huggingface-cli login` or set `HF_TOKEN` env var.
- The notebook will try Gemma first and fall back to a tiny model if access fails.


## 1) Locate PMData participant and files
We'll use participant `p16` as an example (you can change `participant_id`).

In [2]:
pmdata_root = 'medalpaca/data/pmdata'
participant_id = 'p16'  # change to another id if desired
base = os.path.join(pmdata_root, participant_id)
fitbit_dir = os.path.join(base, 'fitbit')
pmsys_dir = os.path.join(base, 'pmsys')

paths = {
    'exercise': os.path.join(fitbit_dir, 'exercise.json'),
    'sleep': os.path.join(fitbit_dir, 'sleep.json'),
    'resting_hr': os.path.join(fitbit_dir, 'resting_heart_rate.json'),
    'wellness': os.path.join(pmsys_dir, 'wellness.csv'),
}
paths

{'exercise': 'medalpaca/data/pmdata/p16/fitbit/exercise.json',
 'sleep': 'medalpaca/data/pmdata/p16/fitbit/sleep.json',
 'resting_hr': 'medalpaca/data/pmdata/p16/fitbit/resting_heart_rate.json',
 'wellness': 'medalpaca/data/pmdata/p16/pmsys/wellness.csv'}

## 2) Read raw sensor and label data
We will parse: 
- Fitbit `exercise.json` (per-activity logs with calories and steps)
- Fitbit `sleep.json` (minutes asleep per sleep period)
- Fitbit `resting_heart_rate.json` (daily RHR)
- PMSys `wellness.csv` (daily self-report with `stress`, `mood`, etc.)

In [3]:
def read_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, 'r') as f:
        return json.load(f)

def read_wellness_csv(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, 'r') as f:
        reader = csv.DictReader(f)
        for r in reader:
            # Parse ISO timestamp to date
            dt = datetime.fromisoformat(r['effective_time_frame'].replace('Z',''))
            rows.append({
                'date': dt.date(),
                'mood': int(r['mood']) if r['mood'] else None,
                'stress': int(r['stress']) if r['stress'] else None,
                'sleep_quality': int(r['sleep_quality']) if r['sleep_quality'] else None,
                'readiness': int(r['readiness']) if r['readiness'] else None,
                'fatigue': int(r['fatigue']) if r['fatigue'] else None,
                'sleep_duration_h': float(r['sleep_duration_h']) if r['sleep_duration_h'] else None,
            })
    return rows

exercise = read_json_list(paths['exercise'])
sleep = read_json_list(paths['sleep'])
resting_hr = read_json_list(paths['resting_hr'])
wellness = read_wellness_csv(paths['wellness'])

print('exercise entries:', len(exercise))
print('sleep entries:', len(sleep))
print('resting_hr entries:', len(resting_hr))
print('wellness entries:', len(wellness))
print('sample wellness row:', wellness[0])

exercise entries: 19
sleep entries: 153
resting_hr entries: 152
wellness entries: 82
sample wellness row: {'date': datetime.date(2019, 11, 15), 'mood': 4, 'stress': 4, 'sleep_quality': 3, 'readiness': 5, 'fatigue': 3, 'sleep_duration_h': 6.0}


## 3) Build 14-day context features aligned to wellness dates
For each wellness log date, we collect the previous 14 days' Fitbit context:
- Steps and calories from `exercise.json`
- Minutes asleep from `sleep.json`
- Resting heart rate from `resting_heart_rate.json`

We use the wellness `stress` (1–5) as the label, and `mood` (1–5) as extra context.

In [4]:
# Normalize raw lists to (date, value) series for simple filtering
def parse_exercise_series(ex):
    out = []
    for e in ex:
        dt = datetime.strptime(e['startTime'], '%Y-%m-%d %H:%M:%S')
        out.append({
            'dt': dt,
            'date': dt.date(),
            'steps': float(e.get('steps', 0.0) or 0.0),
            'calories': float(e.get('calories', 0.0) or 0.0),
        })
    return out

def parse_sleep_series(sl):
    out = []
    for s in sl:
        # Use minutesAsleep as a daily value associated to the end date
        # endTime can be ISO-like or space-separated, handle both
        end_raw = s.get('endTime')
        try:
            dt = datetime.fromisoformat(end_raw.replace('Z',''))
        except Exception:
            dt = datetime.strptime(end_raw, '%Y-%m-%dT%H:%M:%S.%f') if 'T' in end_raw else datetime.strptime(end_raw, '%Y-%m-%d %H:%M:%S')
        out.append({
            'dt': dt,
            'date': dt.date(),
            'minutes_asleep': float(s.get('minutesAsleep', 0.0) or 0.0),
        })
    return out

def parse_rhr_series(rhr):
    out = []
    for r in rhr:
        dt = datetime.strptime(r['dateTime'], '%Y-%m-%d %H:%M:%S')
        val = r.get('value', {})
        out.append({
            'dt': dt,
            'date': dt.date(),
            'rhr': float(val.get('value', 0.0) or 0.0),
        })
    return out

ex_series = parse_exercise_series(exercise)
sl_series = parse_sleep_series(sleep)
rhr_series = parse_rhr_series(resting_hr)

# Index by date for quick slicing
def slice_window(series: List[Dict[str, Any]], key: str, end_dt: datetime, days: int = 14) -> List[float]:
    start_dt = end_dt - timedelta(days=days)
    vals = []
    for item in series:
        if start_dt <= item['dt'] < end_dt:
            vals.append(float(item[key]))
    return vals

print('Example slices around first wellness date:')
end_dt = datetime.combine(wellness[0]['date'], datetime.min.time())
print('steps (len):', len(slice_window(ex_series, 'steps', end_dt)))
print('calories (len):', len(slice_window(ex_series, 'calories', end_dt)))
print('minutes_asleep (len):', len(slice_window(sl_series, 'minutes_asleep', end_dt)))
print('rhr (len):', len(slice_window(rhr_series, 'rhr', end_dt)))

Example slices around first wellness date:
steps (len): 0
calories (len): 0
minutes_asleep (len): 15
rhr (len): 14


## 4) Compose prompts (instruction, input, output)
We follow the repository's style for PMData: encode a 14-day window of steps, calories, resting heart rate, sleep minutes, and include the current mood. Label is the self-reported `stress` (1–5).

In [5]:
examples = []
for row in wellness:
    if row['stress'] is None or row['mood'] is None:
        continue
    end_dt = datetime.combine(row['date'], datetime.min.time())
    steps_hist = slice_window(ex_series, 'steps', end_dt)
    cals_hist = slice_window(ex_series, 'calories', end_dt)
    sleep_hist = slice_window(sl_series, 'minutes_asleep', end_dt)
    rhr_hist = slice_window(rhr_series, 'rhr', end_dt)

    # require at least a few points to make a meaningful example
    if min(len(steps_hist), len(cals_hist), len(sleep_hist), len(rhr_hist)) < 3:
        continue

    instruction = (
        'You are a personalized healthcare agent trained to predict stress '
        'which ranges from 1 to 5 based on physiological data and user information.'
    )
    input_txt = (
        f"The recent 14-days sensor readings show: "
        f"[Steps]: {steps_hist} steps, "
        f"[Burned Calories]: {cals_hist} calories, "
        f"[Resting Heart Rate]: {rhr_hist} beats/min, "
        f"[SleepMinutes]: {sleep_hist} minutes, "
        f"[Mood]: {row['mood']} out of 5; "
        f"What would be the predicted stress?"
    )
    output = str(row['stress'])
    examples.append({'instruction': instruction, 'input': input_txt, 'output': output})

len(examples)

17

In [6]:
# Show a few prompts and labels
for i in range(min(2, len(examples))):
    print(f'Example {i+1}:')
    print('Instruction:', examples[i]['instruction'])
    print('Input:', examples[i]['input'][:300] + ('...' if len(examples[i]['input'])>300 else ''))
    print('Label:', examples[i]['output'])
    print('-'*80)

Example 1:
Instruction: You are a personalized healthcare agent trained to predict stress which ranges from 1 to 5 based on physiological data and user information.
Input: The recent 14-days sensor readings show: [Steps]: [905.0, 2910.0, 2223.0] steps, [Burned Calories]: [91.0, 264.0, 169.0] calories, [Resting Heart Rate]: [66.6323356628418, 65.75046253204346, 66.47550773620605, 66.87107944488525, 66.51373672485352, 66.09467315673828, 63.21689224243164, 62.88582992553...
Label: 3
--------------------------------------------------------------------------------
Example 2:
Instruction: You are a personalized healthcare agent trained to predict stress which ranges from 1 to 5 based on physiological data and user information.
Input: The recent 14-days sensor readings show: [Steps]: [905.0, 2910.0, 2223.0] steps, [Burned Calories]: [91.0, 264.0, 169.0] calories, [Resting Heart Rate]: [65.75046253204346, 66.47550773620605, 66.87107944488525, 66.51373672485352, 66.09467315673828, 63.2168922424

## 5) Tokenization via DataHandler (medAlpaca template)
We use `medalpaca/prompt_templates/medalpaca.json` to format the final prompt string.

In [7]:
# Prefer Gemma-3-270M; fall back if access/download fails.
base_model_name = 'google/gemma-3-270m'
try:
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True, trust_remote_code=True)
except Exception as e:
    print('Falling back to tiny-gpt2 due to:', e)
    base_model_name = 'sshleifer/tiny-gpt2'
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = getattr(tokenizer, 'eos_token_id', 0) or 0
tokenizer.padding_side = 'left'

handler = DataHandler(
    tokenizer=tokenizer,
    prompt_template='medalpaca/prompt_templates/medalpaca.json',
    model_max_length=256,
    train_on_inputs=True,
)

# Inspect one fully rendered prompt and tokenization
sample = examples[0]
prompt = handler.generate_prompt(sample['instruction'], sample['input'], sample['output'])
print(prompt[:600] + ('...' if len(prompt)>600 else ''))
tok = handler.tokenize(prompt)
print('Tokenized keys:', list(tok.keys()))
print('input_ids length:', len(tok['input_ids']))
print('First 20 token ids:', tok['input_ids'][:20])
print('Decoded preview:', tokenizer.decode(tok['input_ids'][:80]))


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a personalized healthcare agent trained to predict stress which ranges from 1 to 5 based on physiological data and user information.

### Input:
The recent 14-days sensor readings show: [Steps]: [905.0, 2910.0, 2223.0] steps, [Burned Calories]: [91.0, 264.0, 169.0] calories, [Resting Heart Rate]: [66.6323356628418, 65.75046253204346, 66.47550773620605, 66.87107944488525, 66.51373672485352, 66.09467315673828, 63.2...
Tokenized keys: ['input_ids', 'attention_mask', 'labels']
input_ids length: 256
First 20 token ids: [43760, 563, 614, 14787, 600, 15517, 496, 4209, 236764, 33481, 607, 614, 2744, 600, 4728, 3342, 4403, 236761, 15642, 496]
Decoded preview: Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

## 6) Create HF Dataset and map tokenization
We convert the examples to a `datasets.Dataset` and apply `generate_and_tokenize_prompt`.

In [8]:
ds = Dataset.from_list(examples)
tokenized_ds = ds.map(handler.generate_and_tokenize_prompt)
tokenized_ds = tokenized_ds.remove_columns([c for c in tokenized_ds.column_names if c not in ('input_ids','attention_mask','labels')])
tokenized_ds

Map: 100%|██████████| 17/17 [00:00<00:00, 1366.41 examples/s]


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 17
})

## 6.5) Zero-shot and Few-shot Inference (No Fine-Tuning)
We evaluate Gemma-3-270M (or the fallback model) without any fine-tuning.
- Zero-shot: Directly prompt with the 14-day context; ask for a single integer 1–5.
- Few-shot: Provide 3 labeled examples, then ask for the answer to a new question.
Note: This is a quick qualitative check; no strict scoring is computed here.


In [9]:

import random

# Ensure model is loaded (prefer Gemma; fallback handled as before)
try:
    model  # noqa: F821
except NameError:
    try:
        model = AutoModelForCausalLM.from_pretrained(base_model_name, trust_remote_code=True, attn_implementation='eager')
    except Exception as e:
        print('Model load failed for', base_model_name, '-> falling back:', e)
        base_model_name = 'sshleifer/tiny-gpt2'
        model = AutoModelForCausalLM.from_pretrained(base_model_name)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

def generate_text(prompt, max_new_tokens=32, temperature=0.7, top_p=0.95):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=top_p)
    return tokenizer.decode(out[0])

def build_zero_shot(ex):
    return (
        'You are a health assistant. Predict the daily stress level based on the context.'
        'Respond with a single integer from 1 to 5.'
        f"Context: {ex['input']}"
        'Answer: '
    )

def build_few_shot(target, shots):
    header = ('You are a health assistant. Read the examples and predict the daily stress.'
              'Respond with a single integer from 1 to 5.')
    blocks = []
    for i, ex in enumerate(shots, 1):
        blocks.append(f"[Example {i}] Question: {ex['input']} Answer: {ex['output']}")
    final = f"Finally, answer the question: Question: {target['input']} Answer: "
    return header + ' '.join(blocks) + ' ' + final

# Select a small evaluation slice
rng = random.Random(0)
eval_ids = rng.sample(range(len(examples)), k=min(5, len(examples))) if len(examples)>0 else []
for idx in eval_ids:
    ex = examples[idx]
    # Zero-shot
    zsp = build_zero_shot(ex)
    zs_out = generate_text(zsp)
    print('--- Zero-shot ---')
    print('Prompt:', zsp[:300] + ('...' if len(zsp)>300 else ''))
    print('Response:', zs_out.split('Answer:')[-1].strip())
    print('Label:', ex['output'])
    # Few-shot
    pool = [i for i in range(len(examples)) if i != idx]
    shot_ids = rng.sample(pool, k=min(3, len(pool))) if pool else []
    shots = [examples[sid] for sid in shot_ids]
    fsp = build_few_shot(ex, shots)
    fs_out = generate_text(fsp)
    print('--- Few-shot (3) ---')
    print('Prompt:', fsp[:300] + ('...' if len(fsp)>300 else ''))
    print('Response:', fs_out.split('Answer:')[-1].strip())
    print('Label:', ex['output'])
    print()


--- Zero-shot ---
Prompt: You are a health assistant. Predict the daily stress level based on the context.Respond with a single integer from 1 to 5.Context: The recent 14-days sensor readings show: [Steps]: [991.0, 993.0, 951.0] steps, [Burned Calories]: [94.0, 87.0, 96.0] calories, [Resting Heart Rate]: [66.49956130981445, ...
Response: 3 out of 5.<eos><eos><eos>2. The 14-days sensor readings show: 39.9% of the respondents are under
Label: 2
--- Few-shot (3) ---
Prompt: You are a health assistant. Read the examples and predict the daily stress.Respond with a single integer from 1 to 5.[Example 1] Question: The recent 14-days sensor readings show: [Steps]: [0.0, 2571.0, 2067.0] steps, [Burned Calories]: [62.0, 199.0, 173.0] calories, [Resting Heart Rate]: [66.502650...
Response: 3 [Example 4] Question: The recent 14-days sensor readings show: [Steps]: [991.0, 99
Label: 2

--- Zero-shot ---
Prompt: You are a health assistant. Predict the daily stress level based on the context.Respond w

In [10]:
import re, math

def parse_pred_to_int(text):
    seg = text.split('Answer:')[-1] if 'Answer:' in text else text
    m = re.search(r"[-+]?\d*\.?\d+", seg)
    if not m:
        return None
    try:
        val = float(m.group(0))
        iv = int(round(val))
        return max(1, min(5, iv))
    except Exception:
        return None

def compute_metrics(preds, gts):
    pairs = [(p, g) for p, g in zip(preds, gts) if p is not None and g is not None]
    if not pairs:
        return {"n": 0, "exact_match": None, "mae": None}
    n = len(pairs)
    em = sum(1 for p, g in pairs if p == g) / n
    mae = sum(abs(p - g) for p, g in pairs) / n
    return {"n": n, "exact_match": em, "mae": mae}

zero_preds, zero_gts = [], []
few_preds, few_gts = [], []

# Re-run the same eval_ids slice for metrics
for idx in eval_ids:
    ex = examples[idx]
    # Zero-shot
    zsp = build_zero_shot(ex)
    zs_out = generate_text(zsp)
    zp = parse_pred_to_int(zs_out)
    try:
        zg = int(ex['output'])
    except Exception:
        zg = None
    zero_preds.append(zp); zero_gts.append(zg)

    # Few-shot (3)
    pool = [i for i in range(len(examples)) if i != idx]
    shot_ids = rng.sample(pool, k=min(3, len(pool))) if pool else []
    shots = [examples[sid] for sid in shot_ids]
    fsp = build_few_shot(ex, shots)
    fs_out = generate_text(fsp)
    fp = parse_pred_to_int(fs_out)
    few_preds.append(fp); few_gts.append(zg)

zs_metrics = compute_metrics(zero_preds, zero_gts)
fs_metrics = compute_metrics(few_preds, few_gts)
print('Zero-shot ->', zs_metrics)
print('Few-shot (3) ->', fs_metrics)


Zero-shot -> {'n': 5, 'exact_match': 0.4, 'mae': 0.8}
Few-shot (3) -> {'n': 5, 'exact_match': 0.6, 'mae': 0.4}


## 7) Minimal Trainer setup
This mirrors `medalpaca/train.py` at a small scale for demonstration.
Switch `base_model_name` to your target model if you want to train seriously.

In [11]:
try:
    model = AutoModelForCausalLM.from_pretrained(base_model_name, trust_remote_code=True)
except Exception as e:
    print('Model load failed for', base_model_name, '-> falling back:', e)
    base_model_name = 'sshleifer/tiny-gpt2'
    model = AutoModelForCausalLM.from_pretrained(base_model_name)

collator = DataCollatorForSeq2Seq(
    tokenizer, pad_to_multiple_of=8, return_tensors='pt', padding=True
)

training_args = TrainingArguments(
    output_dir='output/demo-stress-tiny',
    per_device_train_batch_size=2,
    num_train_epochs=1,
    learning_rate=5e-5,
    logging_steps=5,
    save_steps=50,
    eval_strategy='no',
    fp16=False,
    bf16=False,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds.select(range(min(32, len(tokenized_ds)))),
    data_collator=collator,
)

print(trainer)
print('TrainingArguments summary:', training_args)


TrainingArguments summary: TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=no,
eval_use_gather_obje

### Optional: run a very short training step
Note: This is only to verify the pipeline end-to-end with a tiny model.
For actual experiments, use `medalpaca/train.py` with your target model and proper resources.

In [12]:
# Uncomment to run a quick demo train (may take a minute)
trainer.train()
trainer.save_model()
print('Ready to train: call trainer.train() when you are ready.')

It is strongly recommended to train Gemma3 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.


Step,Training Loss
5,2.187400


Ready to train: call trainer.train() when you are ready.


## 8) Generate a prediction example (post-training or base model)
We reuse the same prompt template but omit the ground-truth `output` to simulate inference.

In [14]:
sample = examples[min(1, len(examples)-1)]
inference_prompt = handler.generate_prompt(sample['instruction'], sample['input'], output=None)
inputs = tokenizer(inference_prompt, return_tensors='pt').to(device)
with torch.no_grad():
    out = model.generate(**{k:v for k,v in inputs.items()}, max_new_tokens=32)
print(tokenizer.decode(out[0]))

<bos>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a personalized healthcare agent trained to predict stress which ranges from 1 to 5 based on physiological data and user information.

### Input:
The recent 14-days sensor readings show: [Steps]: [905.0, 2910.0, 2223.0] steps, [Burned Calories]: [91.0, 264.0, 169.0] calories, [Resting Heart Rate]: [65.75046253204346, 66.47550773620605, 66.87107944488525, 66.51373672485352, 66.09467315673828, 63.21689224243164, 62.88582992553711, 63.4669303894043, 64.84911155700684, 65.60495948791504, 65.19553184509277, 64.71516513824463, 64.67836952209473, 65.35145664215088] beats/min, [SleepMinutes]: [478.0, 401.0, 401.0, 527.0, 460.0, 696.0, 389.0, 321.0, 326.0, 505.0, 495.0, 402.0, 362.0, 504.0] minutes, [Mood]: 3 out of 5; What would be the predicted stress?

### Response:
You are a personalized healthcare agent t

In [20]:
out[0].shape

torch.Size([563])